# Lab 04 · Preparing data for machine learning

Practise cleaning, encoding, and scaling data to prepare it for machine learning.

Fill each `____` using the comments, then run the cells in order.


## Imports


In [ ]:
# Import numpy, pandas, matplotlib.pyplot, StringIO from io, and pickle.
# sklearn tools: train_test_split, SimpleImputer, StandardScaler, MinMaxScaler.
import ____ as np
import ____ as pd
import ____ as plt
from sklearn.model_selection import ____
from sklearn.impute import ____
from sklearn.preprocessing import ____, ____
from io import ____
import ____


## DataFrame Inspection


In [ ]:
# Invented food orders; delivery_minutes is the future prediction target.
orders = pd.____({
    "km": ____,  # Distances: 1,2,4,3,5,2,6,4,1,3,5,2.
    "items": ____,  # Counts: 2,3,4,2,5,1,3,2,4,3,2,5.
    "prep_minutes": ____,  # 8,12,15,None,18,7,16,14,10,13,17,None; None means unrecorded.
    "bag": ____,  # small,medium,large,small,large,small,medium,small,large,medium,small,large.
    "payment": ____,  # cash,card,card,cash,card,cash,card,cash,card,cash,card,cash.
    "delivery_minutes": ____  # Times: 18,25,39,28,48,20,49,36,24,31,42,30.
})
print(orders.____(3))
orders.____()
print(orders.____().sum())


In [ ]:
# Check category frequencies and the ranges of numeric measurements.
print(orders["bag"].____())
print(orders.____().round(2))
orders[["km", "items", "prep_minutes", "delivery_minutes"]].____(bins=4, figsize=(7, 5))
plt.____()
plt.____()


## Scatter Plots


In [ ]:
# Plot orders with recorded preparation time; the two unrecorded times are omitted.
# Dot size shows item count; colour shows preparation minutes.
plt.____(figsize=(6, 4))
points = plt.____(orders["km"], orders["delivery_minutes"], s=orders["items"] * 25,
    c=orders[____], cmap="viridis", alpha=0.7)  # Colour by preparation time.
plt.____(points, label="Preparation time (minutes)")
plt.____("Distance (km)")
plt.____("Delivery time (minutes)")
plt.____()


## Train-Test Split


In [ ]:
# Remove the target from inputs before splitting both together; keep matching indices.
X = orders.____(columns="delivery_minutes")
y = orders[____].copy()
X_train, X_test, y_train, y_test = ____(X, y, test_size=0.25, random_state=17)
print(X_train.____, X_test.shape)
print(X_train.index.____(y_train.index))
# Explore numeric associations using training rows only.
train_numeric = X_train.____(include="number")
print(train_numeric.____().round(2))


## Missing Values


In [ ]:
# A separate receipt log illustrates three choices; preserve the original table.
receipts = pd.____({"items": [2, 4, 3], "discount_aed": [0, None, 2]})
complete_receipts = receipts.____(subset=["discount_aed"])
without_discount = receipts.____(columns="discount_aed")
filled_receipts = receipts.____()
filled_receipts["discount_aed"] = receipts["discount_aed"].____(receipts["discount_aed"].median())
print(complete_receipts.____, without_discount.shape, filled_receipts.shape)
# Compare information lost by deleting a row, deleting a column, or filling a value.


In [ ]:
# Fit on training numeric columns; carry the fitted medians to held-out rows.
numeric_columns = ____  # "km", "items", "prep_minutes", in this order.
imputer = ____(strategy="median")
train_array = imputer.____(X_train[numeric_columns])
test_array = imputer.____(X_test[numeric_columns])
train_filled = pd.____(train_array, columns=numeric_columns, index=X_train.index)
test_filled = pd.____(test_array, columns=numeric_columns, index=X_test.index)
print(train_filled.____())


## Categorical Encoding


In [ ]:
# Bag ranks: small=1, medium=2, large=3. Payment methods have no such order.
bag_order = ____
train_bags = X_train["bag"].____(bag_order).rename("bag_rank")
test_bags = X_test["bag"].____(bag_order).rename("bag_rank")
train_payment = pd.____(X_train["payment"], prefix="pay", dtype=int)
test_payment = pd.____(X_test["payment"], prefix="pay", dtype=int)
test_payment = test_payment.____(columns=train_payment.columns, fill_value=0)
print(____)
# Every payment category in this toy test set also appears in training.


## Feature Scaling


In [ ]:
# Standardize numeric inputs using training statistics.
standard = ____()
train_z = pd.____(standard.fit_transform(train_filled), columns=numeric_columns, index=X_train.index)
test_z = pd.____(standard.transform(test_filled), columns=numeric_columns, index=X_test.index)
print(train_z.____().round(6))
print(train_z.____(ddof=0).round(6))
# ddof=0 matches StandardScaler's variance convention.


In [ ]:
# Compare an alternative numeric representation on a 0-to-1 training scale.
minmax = ____()
train_01 = pd.____(minmax.fit_transform(train_filled), columns=numeric_columns, index=X_train.index)
test_01 = pd.____(minmax.transform(test_filled), columns=numeric_columns, index=X_test.index)
print(train_01.____(), train_01.max())
print(test_01.____(), test_01.max())
# Held-out values can lie outside the training range; do not refit to force them inside.


## Feature Alignment


In [ ]:
# Join each order's scaled numbers, bag rank, and payment indicators by row index.
prepared_train = pd.____([train_z, train_bags, train_payment], axis=1)
prepared_test = pd.____([test_z, test_bags, test_payment], axis=1)
print(prepared_train.index.____(y_train.index))
print(prepared_test.index.____(y_test.index))
print(prepared_train.____().sum())
# These are prepared inputs, not predictions or evidence of model accuracy.


## Pickle Serialization


In [ ]:
# Round-trip our own fitted objects in memory; no extra files are required.
bundle = ____  # Keys: imputer, scaler, numeric_columns; values: imputer, standard, numeric_columns.
saved_bytes = pickle.____(bundle)
restored = pickle.____(saved_bytes)    # Load only the bytes just created here.
# A new order has 3 km, 2 items, and an unrecorded preparation time.
new_order = pd.____([[3, 2, np.nan]], columns=restored["numeric_columns"])
new_filled = restored["imputer"].____(new_order)
new_filled = pd.____(new_filled, columns=restored["numeric_columns"])
print(restored["scaler"].____(new_filled))


## CSV Export


In [ ]:
# Store features and their target together so row correspondence survives CSV export.
export_table = prepared_train.____()
export_table["delivery_minutes"] = ____
csv_text = export_table.____(index=False)
reloaded = pd.____(StringIO(csv_text))
print(reloaded.____(2))
print(reloaded.____)
# csv_text can be saved later; this practice keeps it in memory.


## Capped Values


In [ ]:
# A phone app records walks longer than 60 minutes as 60. Keep that rule visible.
walk_log = pd.____({"recorded_minutes": [20, 45, 60, 35, 60, 60]})
at_cap = walk_log["recorded_minutes"] ____ 60
print(at_cap.____())
print(walk_log["recorded_minutes"].____())
# The recorded mean may understate actual walking time; values of 60 can hide longer walks.
